8 вариант

Модель: MMS tts

Язык: armenian  

https://huggingface.co/facebook/mms-tts-hyw


Критерии для текстов:

- а) Каждое высказывание должно быть логичным предложением (начинается с заглавной буквы и заканчивается точкой).
- б) Минимальная длина — 7 слов.
- в) Максимальная длина — 70 слов (можно больше, но не рекомендуется).
- г) Цифры и аббревиатуры записываются словами:

не "193", а "сто девяносто три";

не "ФГУП", а "федеральное государственное унитарное предприятие".

группа_фамилия_вариант_модель_язык.tar.gz
bvt2201_ivanov_17_espeak_ru.tar.gz

книга 1  ===> https://psv4.userapi.com/s/v1/d/7kb1WBiCUBRw2QrEzyVhJSXhg5AHYpbUTyk7qiacIxwEmefmbZtScDygOIAu4J1omog74Yr0bkAova-WLCWud5A38CThhSDkqVCjXQ5J0PhteaZe-GYbjw/16_2017_Mezhpartiynye_otnoshenia_v_Respublike_Armenia_v_1918-1920.pdf

In [1]:
pip install --upgrade transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [2]:
from transformers import VitsModel, AutoTokenizer
import torch
from IPython.display import Audio
import scipy
import numpy as np
import os
from pathlib import Path

In [3]:
# init model
model = VitsModel.from_pretrained("facebook/mms-tts-hyw")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-hyw")
model.to(device)

text = "Նարէ՜, աս տա՜ր"

def test_tts(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    if inputs['input_ids'].shape[1] == 0:
        print("Warning: Tokenizer returned empty sequence")
        return None

    print(f"Input sequence length: {inputs['input_ids'].shape[1]}")

    with torch.no_grad():
        try:
            output = model(**inputs).waveform
            return output
        except Exception as e:
            print(f"Error during inference: {e}")
            return None

output = test_tts(text, model, tokenizer)

if output is not None:
    print(f"Successful generated audio with {len(output[0])} samples")

# audio_numpy = output.cpu().numpy()
Audio(output.cpu().numpy(), rate=model.config.sampling_rate)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

Input sequence length: 23
Successful generated audio with 30720 samples


In [4]:
def save_wav(output, filename="test.wav"):
  # тензор PyTorch в numpy array
  audio_data = output.cpu().numpy()
  if audio_data.ndim == 2 and audio_data.shape[0] == 1:
      audio_data = audio_data[0]

  scipy.io.wavfile.write(filename, rate=model.config.sampling_rate, data=audio_data)

save_wav(output)

In [ ]:
def synthesize_from_txt(txt_filename, output_dir="/content/drive/MyDrive/output"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    input_path = f"/content/{txt_filename}"

    if not os.path.exists(input_path):
        print(f"File {input_path} not found!")
        return

    with open(input_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    print(f"Found {len(lines)} lines in {txt_filename}")
    successful_syntheses = 0

    for i, line in enumerate(lines, 1):
        text = line.strip()
        if not text:
            continue

        inputs = tokenizer(text, return_tensors="pt")
        inputs = {key: value.to(device) for key, value in inputs.items()}
        with torch.no_grad():
            try:
                output = model(**inputs).waveform

                output_filename = f"{output_dir}/{i}.wav"
                save_wav(output, output_filename)
                successful_syntheses += 1

            except Exception as e:
                print(f"Error processing line {i}: {e}")
                continue

    print(f"\nCompleted! Successfully generated {successful_syntheses} out of {len(lines)} audio files.")

synthesize_from_txt("10k-samples.txt")

File /content/10k-samples.txt not found!


In [6]:
synthesize_from_txt("10k-samples-filtered.txt")

Found 10402 lines in 10k-samples-filtered.txt

Completed! Successfully generated 10402 out of 10402 audio files.
